# 4. Nulling interferometry

Arrays of telescopes are combined by photonic chips. A **Bracewell nuller** sends the light of an on-axis
star to a bright output and transmits $2P\sin^2(\pi B\theta/\lambda)$ from a source at angle
$\theta$ on its dark output (Bracewell 1978). A **kernel nuller** (Martinache & Ireland 2018) combines
four telescopes into observables robust to small phase errors.

In [ ]:
import numpy as np
from astropy import units as u

import lumos
from lumos import plotting
from lumos.elements.photonics import kernels
from lumos.plotting import style

style.apply()

## Transmission of a Bracewell nuller

The preset `bracewell.lumos` holds a source, two telescopes 20 m apart and the nuller. Moving the source
across the field gives the transmission map.

In [ ]:
ctx = lumos.io.load("presets/bracewell.lumos")
lam, baseline = 10e-6, 20.0
thetas = np.linspace(-1.5, 1.5, 61) * lam / baseline
points = ctx.sweep({"source.theta_x": thetas * u.rad}, outputs=["meter.out"])
null = np.array([p.outputs["meter.out"].data.to_value(u.W)[0] for p in points])
bright = np.array([p.outputs["meter.out"].data.to_value(u.W)[1] for p in points])
transmission = null / (null + bright)
fig, ax = style.figure("single")
ax.plot(thetas / (lam / baseline), transmission, label="LUMOS")
ax.plot(thetas / (lam / baseline), np.sin(np.pi * baseline * thetas / lam) ** 2, "k--", label="sin²(πBθ/λ)")
ax.set_xlabel("θ (λ/B)")
ax.set_ylabel("transmission")
ax.legend();

In [ ]:
result = ctx.run(ctx.meta["outputs"], archive="bracewell")
result.archive.save_figure(fig, "transmission", data={"theta": thetas * u.rad, "transmission": transmission});

## Outputs of a kernel nuller

The preset `kernel_nuller.lumos` observes a star and a planet 100 times fainter: output 0 collects the star,
the six dark outputs the planet, and the kernels are differences of pairs of dark outputs.

In [ ]:
ctx = lumos.io.load("presets/kernel_nuller.lumos")
result = ctx.run(ctx.meta["outputs"], archive="kernel-nuller")
power = result["meter.out"]
fig, axes = style.figure("double", ncols=2, aspect=0.4)
plotting.plot_chip_outputs(power, ax=axes[0], labels=["bright"] + [f"dark {k}" for k in range(1, 7)])
axes[0].set_yscale("log")
values = kernels(power.data.to_value(u.W))
axes[1].bar(range(1, 4), values)
axes[1].set_xticks(range(1, 4))
axes[1].set_xlabel("kernel")
axes[1].set_ylabel(style.axis_label("kernel", u.W))
result.archive.save_figure(fig, "kernel_nuller", data={"power": power.data, "kernels": values * u.W})
values